In [6]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

sns.set_style('whitegrid')
print("All imports successful!")

All imports successful!


In [ ]:
import os
from pathlib import Path

# Load the dataset
csv_filename = 'Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv'
csv_path = Path(csv_filename)

if not csv_path.exists():
    print(f"⚠️ Dataset file not found: {csv_filename}")
    print(f"Current notebook directory: {Path.cwd()}")
    print("Available files in current directory:")
    for file in sorted(os.listdir('.')):
        print(f"  {file}")
    raise FileNotFoundError(
        f"Please place '{csv_filename}' in the notebook directory or update the path to the dataset file."
    )

df = pd.read_csv(csv_path)

print("="*60)
print("DATASET OVERVIEW")
print("="*60)
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")
print("\nFirst 5 rows:")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv'

## Dataset Description

This dataset contains smartphone usage patterns and addiction labels from 7,500 users.

### Key Features:
- **Demographics:** age, gender
- **Usage Metrics:** daily_screen_time_hours, social_media_hours, gaming_hours, work_study_hours
- **Behavioral:** notifications_per_day, app_opens_per_day, weekend_screen_time
- **Health:** sleep_hours, stress_level
- **Target:** addicted_label (0 = Not Addicted, 1 = Addicted)

### Goal:
Predict whether a user is addicted to their smartphone based on their usage patterns.

In [ ]:
print("="*60)
print("DATA QUALITY CHECK")
print("="*60)

# Check missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Check data types
print("\nData Types:")
print(df.dtypes)

# Check unique values in categorical columns
categorical = df.select_dtypes(include=['object']).columns
print(f"\nCategorical Columns: {categorical.tolist()}")
for col in categorical:
    print(f"  {col}: {df[col].unique()}")

In [ ]:
print("="*60)
print("TARGET VARIABLE ANALYSIS")
print("="*60)

print(f"\naddicted_label distribution:")
print(df['addicted_label'].value_counts())
print(f"\nPercentage:")
print(df['addicted_label'].value_counts(normalize=True) * 100)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
sns.countplot(data=df, x='addicted_label', ax=axes[0])
axes[0].set_title('Addiction Label Count')
axes[0].set_xlabel('Addicted (1) vs Not Addicted (0)')

# Pie chart
df['addicted_label'].value_counts().plot.pie(autopct='%1.1f%%', ax=axes[1])
axes[1].set_title('Addiction Label Percentage')

plt.tight_layout()
plt.show()

In [ ]:
print("="*60)
print("SUMMARY STATISTICS")
print("="*60)

# Numerical columns only
numerical_cols = df.select_dtypes(include=[np.number]).columns
print(df[numerical_cols].describe())

# Check for any anomalies
print("\nPotential Issues:")
print(f"Minimum age: {df['age'].min()}")
print(f"Maximum daily screen time: {df['daily_screen_time_hours'].max()}")
print(f"Minimum sleep hours: {df['sleep_hours'].min()}")

In [ ]:
print("="*60)
print("EXPLORATORY DATA ANALYSIS - VISUALIZATIONS")
print("="*60)

# Select key numeric features
key_features = ['daily_screen_time_hours', 'social_media_hours', 
                'gaming_hours', 'work_study_hours', 'sleep_hours']

# Create boxplots comparing addiction vs non-addiction
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, feature in enumerate(key_features):
    sns.boxplot(data=df, x='addicted_label', y=feature, ax=axes[i])
    axes[i].set_title(f'{feature} by Addiction Status')
    axes[i].set_xlabel('Addicted (1) vs Not (0)')

# Remove empty subplot
for j in range(len(key_features), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(14, 10))
correlation_matrix = df[numerical_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', 
            linewidths=0.5, square=True)
plt.title('Correlation Heatmap of All Numerical Features')
plt.tight_layout()
plt.show()

# Find top correlations with target
print("\nTop 5 features most correlated with addiction_label:")
corr_with_target = correlation_matrix['addicted_label'].sort_values(ascending=False)
print(corr_with_target.drop('addicted_label').head(10))

## Initial Observations

### Key Findings:
1. **Dataset Size:** 7,500 records, no missing values
2. **Target Distribution:** [Will be visible from the plot]
3. **Key Features for Prediction:**
   - Daily screen time appears higher in addicted users
   - Social media usage shows a clear difference
   - Sleep hours are lower in addicted users

### Next Steps:
1. Encode categorical variables (gender, stress_level, etc.)
2. Split data into training and test sets
3. Train multiple models and compare performance
4. Tune hyperparameters for best model

In [ ]:
print("="*60)
print("DATA PREPARATION FOR MACHINE LEARNING")
print("="*60)

# Encode categorical variables
le = LabelEncoder()
categorical_cols = ['gender', 'stress_level', 'academic_work_impact', 'addiction_level']

for col in categorical_cols:
    df[col + '_encoded'] = le.fit_transform(df[col])
    print(f"Encoded {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Define features
feature_cols = ['age', 'daily_screen_time_hours', 'social_media_hours', 
                'gaming_hours', 'work_study_hours', 'sleep_hours',
                'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time',
                'gender_encoded', 'stress_level_encoded', 
                'academic_work_impact_encoded', 'addiction_level_encoded']

X = df[feature_cols]
y = df['addicted_label']

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
print("="*60)
print("TRAINING MACHINE LEARNING MODELS")
print("="*60)

# Define models to test
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'KNN': KNeighborsClassifier()
}

# Store results
results = {}
predictions = {}

# Train and evaluate each model
for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    results[name] = {
        'accuracy': accuracy,
        'f1_score': f1,
        'model': model
    }
    predictions[name] = y_pred
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  F1 Score: {f1:.4f}")

print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)
comparison_df = pd.DataFrame({
    'Model': results.keys(),
    'Accuracy': [v['accuracy'] for v in results.values()],
    'F1 Score': [v['f1_score'] for v in results.values()]
})
print(comparison_df.to_string(index=False))

TRAINING MACHINE LEARNING MODELS


NameError: name 'LogisticRegression' is not defined

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
models_names = list(results.keys())
accuracy_scores = [results[m]['accuracy'] for m in models_names]

bars = axes[0].bar(models_names, accuracy_scores, color=['blue', 'green', 'red', 'orange'])
axes[0].set_title('Model Accuracy Comparison')
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0, 1)
for bar, score in zip(bars, accuracy_scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                 f'{score:.4f}', ha='center', va='bottom')

# F1 Score comparison
f1_scores = [results[m]['f1_score'] for m in models_names]

bars = axes[1].bar(models_names, f1_scores, color=['blue', 'green', 'red', 'orange'])
axes[1].set_title('F1 Score Comparison')
axes[1].set_xlabel('Model')
axes[1].set_ylabel('F1 Score')
axes[1].set_ylim(0, 1)
for bar, score in zip(bars, f1_scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                 f'{score:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
print("="*60)
print("CONFUSION MATRICES")
print("="*60)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (name, y_pred) in enumerate(predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Not Addicted', 'Addicted'],
                yticklabels=['Not Addicted', 'Addicted'])
    axes[idx].set_title(f'{name}\nAccuracy: {results[name]["accuracy"]:.4f}')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
print("="*60)
print("CLASSIFICATION REPORTS")
print("="*60)

for name, y_pred in predictions.items():
    print(f"\n{name}")
    print("-" * 40)
    print(classification_report(y_test, y_pred, target_names=['Not Addicted', 'Addicted']))

In [ ]:
print("="*60)
print("CROSS-VALIDATION (5-Fold)")
print("="*60)

cv_results = {}

for name, model in models.items():
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    cv_results[name] = cv_scores
    print(f"\n{name}:")
    print(f"  CV Scores: {cv_scores}")
    print(f"  Mean: {cv_scores.mean():.4f}")
    print(f"  Std: {cv_scores.std():.4f}")

# Visualize cross-validation results
plt.figure(figsize=(10, 6))
plt.boxplot([cv_results[name] for name in models_names], labels=models_names)
plt.title('Cross-Validation Scores (5-Fold)')
plt.ylabel('Accuracy')
plt.xlabel('Model')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
print("="*60)
print("HYPERPARAMETER TUNING - RANDOM FOREST")
print("="*60)

# Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Perform Grid Search
rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")

# Evaluate best model on test set
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
best_accuracy = accuracy_score(y_test, y_pred_best)
best_f1 = f1_score(y_test, y_pred_best)

print(f"\nTest Accuracy: {best_accuracy:.4f}")
print(f"Test F1 Score: {best_f1:.4f}")

In [ ]:
print("="*60)
print("FEATURE IMPORTANCE (Random Forest)")
print("="*60)

# Get feature importance from best model
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_rf.feature_importances_
}).sort_values('Importance', ascending=False)

print(feature_importance.to_string(index=False))

# Visualize
plt.figure(figsize=(12, 8))
sns.barplot(data=feature_importance.head(10), x='Importance', y='Feature')
plt.title('Top 10 Most Important Features')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## Results Summary

### Best Performing Model:
[You'll fill this in after seeing results]

### Key Findings:
1. **Most Important Features:**
   - [Fill in from feature importance]
   - [Fill in from feature importance]

2. **Model Performance:**
   - Best Accuracy: [Your best model accuracy]
   - Best F1 Score: [Your best model F1]

3. **What This Means:**
   - [Interpret your results]
   - [Discuss which features are most predictive]

### Limitations:
- [List any limitations of your analysis]
- [Data quality issues if any]
- [Model limitations]

### Future Work:
- [Suggest improvements]
- [What else could be done]

In [ ]:
print("="*60)
print("SAVING RESULTS")
print("="*60)

# Save model performance summary
summary_df = pd.DataFrame({
    'Model': models_names,
    'Accuracy': [results[m]['accuracy'] for m in models_names],
    'F1 Score': [results[m]['f1_score'] for m in models_names],
    'CV Mean': [cv_results[m].mean() for m in models_names],
    'CV Std': [cv_results[m].std() for m in models_names]
})
print("\nSummary Table:")
print(summary_df.to_string(index=False))

# Save to CSV (optional)
summary_df.to_csv('model_results_summary.csv', index=False)
feature_importance.to_csv('feature_importance.csv', index=False)

print("\n✅ Results saved to CSV files")

In [ ]:
print("="*60)
print("MAKING PREDICTIONS ON NEW DATA")
print("="*60)

# Create example new user data (using the test set as example)
new_data = X_test.iloc[:5].copy()
true_labels = y_test.iloc[:5]

# Make predictions
predictions_new = best_rf.predict(new_data)
probabilities = best_rf.predict_proba(new_data)

# Display results
print("\nPredictions for 5 sample users:")
print("-" * 60)
for i in range(len(new_data)):
    print(f"\nUser {i+1}:")
    print(f"  Actual: {'Addicted' if true_labels.iloc[i] == 1 else 'Not Addicted'}")
    print(f"  Predicted: {'Addicted' if predictions_new[i] == 1 else 'Not Addicted'}")
    print(f"  Probability: {probabilities[i][1]:.2f} (Addicted)")

In [ ]:
import pandas as pd

# View model results
model_results = pd.read_csv('model_results_summary.csv')
print(model_results)

FileNotFoundError: [Errno 2] No such file or directory: 'model_results_summary.csv'

In [ ]:
import os

# List all files in the current directory
print("Files in current directory:")
for file in os.listdir('.'):
    print(f"  {file}")

Files in current directory:
  project_analysis.ipynb
  report.md
  venv


In [ ]:
# Display model results from the notebook variables
try:
    import pandas as pd
    
    # Check if variables exist
    if 'results' in dir():
        print("="*60)
        print("MODEL RESULTS")
        print("="*60)
        
        models_names = list(results.keys())
        data = {
            'Model': models_names,
            'Accuracy': [results[m]['accuracy'] for m in models_names],
            'F1 Score': [results[m]['f1_score'] for m in models_names]
        }
        df = pd.DataFrame(data)
        print(df.to_string(index=False))
    else:
        print("⚠️ Please run Cell 11 first!")
        
except NameError:
    print("⚠️ Variables not found. Please run Cell 11 first.")

⚠️ Please run Cell 11 first!
